In [ ]:
import os

from IPython.display import FileLink

if os.getcwd() == '/notebooks':
    os.chdir("./motion-synthesis")
    print('inside dir: ', os.listdir())
print(os.listdir())
print("Click here to download the dit_d0.tar: ", display(FileLink("/notebooks/motion-synthesis/dit_d0.tar")))


os.environ["TOKENIZERS_PARALLELISM"] = "false"

['requirements.txt', 'networks', 'dit_stable_crossattn_debug.tar', 'glove', 'exp_results', 'README.md', 'conda_requirements.txt', 'prepare', 'motionvae_debug.tar', 'data_utils', 'dataset_split_builder.py', 'README_v1.md', 'demo.py', 'checkpoints', 'environment.yaml', 'dit_stable_crossattn_full.tar', 'eval_models', 'environment_cpu.yaml', '.git', 'assets', 'options', 'main.ipynb', 'motion-dataset.zip', 'common', 'lambda_requirements.txt', 'main.py', '.ipynb_checkpoints', 'data', '.gitignore', 'log', 'utils']


/notebooks/motion-synthesis/dit_d0.tar

Click here to download the dit_d0.tar:  None


In [ ]:
import torch
from torch.backends import cuda
from options.train_options  import TrainOptions
from os.path import join as pjoin
import os
from utils.paramUtils import t2m_kinematic_chain
import numpy as np
from utils.word_vectorizer import WordVectorizer
from torch.utils.data import DataLoader
from data_utils.dataset import MotionDataset
from data_utils.dataset import PartMotionDataset
from networks.nn import MotionVQVAE, DiT, MotionVAE
from networks.trainers import MotionVQVAETrainer, MotionDiTTrainer, MotionVAETrainer
from torch.utils.data import Subset
from networks.nn_validator import VAEValidator, DiffusionValidator
import json

2026-07-10 05:54:12.155682: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-07-10 05:54:12.155753: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-07-10 05:54:12.156763: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-07-10 05:54:12.163567: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-10 05:54:12.921101: W tensorflow/compiler/tf2

In [ ]:
parser = TrainOptions()
options = parser.parse(args = ['--stage', 'autoencoder', '--max_epoch', '100', '--kl_beta_max','0.001', '--kl_beta_cycles', '8', '--lr', '1e-4', '--save_latest', '50', '--eval_every_e', '1', '--save_every_e', '50', '--log_every', '1'])
options.gpu_id = torch.cuda.current_device() if torch.cuda.is_available() else -1
options.device = torch.device("cpu" if options.gpu_id==-1 else "cuda:" + str(options.gpu_id))
torch.autograd.set_detect_anomaly(True)

# disabling flash backend till the architecture parameters allow stable use of flash backend for attention
# without creating nans
cuda.enable_flash_sdp(False)
cuda.enable_mem_efficient_sdp(False)
cuda.enable_math_sdp(True)

if options.gpu_id != -1:
    # self.opt.gpu_id = int(self.opt.gpu_id)
    torch.cuda.set_device(options.gpu_id)

print('\nDevice used: ', options.device)
options.save_root = pjoin(options.checkpoints_dir, 'HumanML3D', options.name)
options.model_dir = pjoin(options.checkpoints_dir, 'model')
options.meta_dir = pjoin(options.save_root, 'meta')
options.eval_dir = pjoin(options.save_root, 'animation')
options.log_dir = pjoin('./log', options.dataset_name, options.name)
options.experiment_dir = './exp_results/vae-setup/finetune_vae'
#options.experiment_dir = './exp_results/stable-diffusion-setup/pipeline_1a_clip_tsg_ca'
options.output_dir = options.experiment_dir
options.is_train = True
options.is_continue = True
options.dataset_mode = "full"
options.batch_size = 128
#options.model_filename = 'dit_stable_crossattn_full.tar'
options.model_filename = 'motionvae_debug.tar'
os.makedirs(options.model_dir, exist_ok=True)
os.makedirs(options.meta_dir, exist_ok=True)
os.makedirs(options.eval_dir, exist_ok=True)
os.makedirs(options.log_dir, exist_ok=True)

options.data_root = './data/HumanML3D'
options.motion_dir = pjoin(options.data_root, 'new_joint_vecs')
options.text_dir = pjoin(options.data_root, 'texts')
options.joints_num = 22
options.max_motion_length = 120
dim_pose = 263
radius = 4
fps = 20
kinematic_chain = t2m_kinematic_chain

print('Set parameters')
print('Learning rate: ', options.lr)
print('Max epochs: ', options.max_epoch)
print('Save latest frequency: ', options.save_latest)
print('Save every epoch frequency: ', options.save_every_e)
print('Log every iterations frequency: ', options.log_every)


Device used:  cuda:0
Set parameters
Learning rate:  0.0001
Max epochs:  100
Save latest frequency:  50
Save every epoch frequency:  50
Log every iterations frequency:  1


In [ ]:
mean = np.load(pjoin(options.data_root, 'Mean.npy'))
std = np.load(pjoin(options.data_root, 'Std.npy'))

w_vectorizer = WordVectorizer('./glove', 'our_vab')
train_split_fn = 'train.txt'
val_split_fn = 'val.txt'
test_split_fn = 'test.txt'
simple_test_split_fn = 'simple_test.txt'

if options.dataset_mode in ["debug", "nano", "micro"]:
    train_split_fn = f'train_{options.dataset_mode}.txt'
    val_split_fn = f'val_{options.dataset_mode}.txt'

train_split_file = pjoin(options.data_root, train_split_fn)
val_split_file = pjoin(options.data_root, val_split_fn)
test_split_file = pjoin(options.data_root, test_split_fn)
simple_test_split_file = pjoin(options.data_root, simple_test_split_fn)

train_dataset = PartMotionDataset(options, mean, std, train_split_file, w_vectorizer)
val_dataset = PartMotionDataset(options, mean, std, val_split_file, w_vectorizer)
if not(options.is_train):
    test_dataset = PartMotionDataset(options, mean, std, test_split_file, w_vectorizer)

print('\nTotal number of snippets in train: ', len(train_dataset))
print('Total number of snippets in val: ', len(val_dataset))
if not(options.is_train):
    print('Total number of snippets in test: ', len(test_dataset))

sample_motion = train_dataset[4]
print('Sample data shape: ', sample_motion['motion_parts'].shape, sample_motion['text'])
Dp_max = sample_motion['motion_parts'].shape[-1]

id list 23384


100%|██████████| 23384/23384 [09:48<00:00, 39.73it/s]


Motion shape (B, T, D): (69896, 120, 263)
Total number of motions 69896
Total number of small motions: 9284
id list 1460


100%|██████████| 1460/1460 [00:35<00:00, 40.88it/s]

Motion shape (B, T, D): (4362, 120, 263)
Total number of motions 4362
Total number of small motions: 550

Total number of snippets in train:  69896
Total number of snippets in val:  4362
Sample data shape:  (120, 6, 60) a man full-body sideways jumps to his left.


In [5]:
train_loader = DataLoader(train_dataset, batch_size=options.batch_size, drop_last=not(options.dataset_mode in ['micro', 'nano']), num_workers=1,
                              shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=options.batch_size, drop_last=not(options.dataset_mode in ['micro', 'nano']), num_workers=1,
                        shuffle=True, pin_memory=True)
if not(options.is_train):
    test_loader = DataLoader(test_dataset, batch_size=options.batch_size, drop_last=not(options.dataset_mode in ['micro', 'nano']), num_workers=1,
                             shuffle=True, pin_memory=True)

if options.stage == "autoencoder":
    '''
    vqvae = MotionVQVAE(
        input_dim=Dp_max,
        enc_hidden_dim=1024,
        dec_hidden_dim=1024,
        latent_dim=256,
        num_embeddings=512,
        beta=0.1
    )

    if options.is_train:
        trainer = MotionVQVAETrainer(options, vqvae = vqvae)
        trainer.train(
            train_dataloader=train_loader,
            val_dataloader=val_loader)
    '''
    vae = MotionVAE(
        dim = 263, #input dimension of motion vector
        hidden_size = 512, # latent dimension
        max_seq_len=options.max_motion_length,
        num_heads = 4,
        depth = 9
    )
    if options.is_train:
        trainer = MotionVAETrainer(options, vae = vae)
        trainer.train(
            train_dataloader=train_loader,
            val_dataloader=val_loader
        )
else:
    dit = DiT(
        input_size = 512, # latent dimension
        hidden_size = 1152,
        text_dim = 768,
        max_seq_len=options.max_motion_length // 4
    )

    if options.is_train:
        trainer = MotionDiTTrainer(
            args = options,
            dit = dit,
            autoencoder_type="pretrained_vae"
        )
        trainer.train(
            train_dataloader=train_loader,
            val_dataloader=val_loader
        )


Number of epochs: 100
Resuming training from previous checkpoint at epoch 31
Iters Per Epoch, Training: 0546, Validation: 034
Epoch: 31
Train Loss: 0.39487 Reconstruction Loss: 0.38922 KL Loss: 28.37773
Validation Loss: 0.41246 Reconstruction Loss: 0.40456 KL Loss: 28.13273
Epoch: 32
Train Loss: 0.39689 Reconstruction Loss: 0.38769 KL Loss: 25.58560
Validation Loss: 0.41613 Reconstruction Loss: 0.40465 KL Loss: 26.05470
Epoch: 33
Train Loss: 0.39861 Reconstruction Loss: 0.38620 KL Loss: 23.87034
Validation Loss: 0.42038 Reconstruction Loss: 0.40572 KL Loss: 24.39732
Epoch: 34
Train Loss: 0.40012 Reconstruction Loss: 0.38474 KL Loss: 22.63146
Validation Loss: 0.42175 Reconstruction Loss: 0.40390 KL Loss: 23.47106
Epoch: 35
Train Loss: 0.40189 Reconstruction Loss: 0.38363 KL Loss: 21.73638
Validation Loss: 0.42539 Reconstruction Loss: 0.40496 KL Loss: 22.18110
Epoch: 36
Train Loss: 0.39141 Reconstruction Loss: 0.38071 KL Loss: 25.97741
Validation Loss: 0.40564 Reconstruction Loss: 0.4029

In [6]:
if options.stage == "autoencoder":
    test_model_filepath = pjoin(options.model_dir, options.model_filename)

    if not(options.is_train) and os.path.exists(test_model_filepath):
        vae_model_dict = torch.load(test_model_filepath, map_location = options.device)

        vae.load_state_dict(vae_model_dict['vae'])

        vae_validator = VAEValidator(
            opt = options,
            vae_model=vae,
            model_type = "vae",
            train_dataloader=train_loader,
            val_dataloader = val_loader
        )
        vae_validator.validate()
    else:
        print("Invalid mode or model file doesn't exist!")
else:
    test_model_filepath = pjoin(options.model_dir, options.model_filename)
    if not(options.is_train) and os.path.exists(test_model_filepath):
        dit_model_dict = torch.load(test_model_filepath, map_location = options.device)

        dit.load_state_dict(dit_model_dict['dit'])

        dit_validator = DiffusionValidator(
            opt = options,
            dit=dit,
            val_dataloader = val_loader,
            test_dataloader=test_loader,
            test_type = "val"
        )
        dit_validator.validate()
    else:
        print("Invalid mode or model file doesn't exist!")